# SmolVLA Full Run (6-hour budget)

**Purpose:** Finish P&P detector rollouts on the v2 slice, reusing existing results in [`results_smolvla/rollouts_smolvla.db`](results_smolvla/rollouts_smolvla.db).

**Already done (skip automatically):**
- 80/80 `vanilla` SmolVLA episodes
- Partial `pnp_uncertainty_only` (likely ~16 at `[4,5]` from the 1-hour sample run)

**This notebook:**
1. Mount Drive + restore package snapshot (~2 min)
2. Show coverage (what's missing)
3. Run **only missing** `pnp_uncertainty_only` rollouts until time runs out

**Priority (best detector config first):** `[4,5]` → `[3,4]` → `[2,3]` (same as π0.5 v2).

**Runtime:** Set `MAX_RUNTIME_HOURS = 5.5` (leave ~30 min buffer). At ~5–6 min/P&P episode, expect **~55–65 new episodes** — enough to complete all 80 at `[4,5]` and maybe start other configs.

**After run:** Re-run [`pnp_smolvla_jennifer_analysis.ipynb`](pnp_smolvla_jennifer_analysis.ipynb).

> Prerequisite: `test_pi05_jennifer.ipynb` Section 3 snapshot on Drive (`smolvla_colab_cache/site_packages.tar.gz`).


---
## 1. Drive + paths (every session)


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
SHARED = f'{DRIVE}/cs159-sp26'
CACHE_DIR = f'{DRIVE}/smolvla_colab_cache'
SNAPSHOT = f'{CACHE_DIR}/site_packages.tar.gz'
HF_HOME = f'{CACHE_DIR}/hf_models'

PI05_V2_DB = f'{SHARED}/results_v2/rollouts_v2.db'
SMOLVLA_RESULTS_DIR = f'{SHARED}/results_smolvla'
SMOLVLA_DB = f'{SMOLVLA_RESULTS_DIR}/rollouts_smolvla.db'
SMOLVLA_VIDEO_DIR = f'{SMOLVLA_RESULTS_DIR}/videos_smolvla'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)
os.makedirs(SMOLVLA_RESULTS_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_HOME
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MUJOCO_GL'] = 'egl'

print(f'SmolVLA DB exists: {os.path.isfile(SMOLVLA_DB)}')
print(f'Snapshot exists:   {os.path.isfile(SNAPSHOT)}')


---
## 2. Restore packages (~2 min)

Run once per Colab session. Uses the snapshot from `test_pi05_jennifer.ipynb`.


In [ ]:
import subprocess, sys, os, shutil, importlib

if not os.path.isfile(SNAPSHOT):
    raise FileNotFoundError('No snapshot — run test_pi05_jennifer Section 3 first.')

LOCAL_SNAPSHOT = '/content/site_packages_restore.tar.gz'
print(f'Copying snapshot ({os.path.getsize(SNAPSHOT)/1e6:.0f} MB)...')
shutil.copy(SNAPSHOT, LOCAL_SNAPSHOT)
subprocess.run(['tar', '-xzf', LOCAL_SNAPSHOT, '-C', '/'], check=True)
os.remove(LOCAL_SNAPSHOT)
importlib.invalidate_caches()

import torch
for pkg in ['mujoco', 'libero', 'lerobot']:
    importlib.import_module(pkg)
    print(f'  {pkg} OK')
print(f'torch {torch.__version__}')

subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=False)


---
## 3. Import eval core + load SmolVLA

Uses [`smolvla_eval_core.py`](smolvla_eval_core.py) (same code as `test_smolvla_jennifer.ipynb`).


In [ ]:
import sys, time
from itertools import groupby
from tqdm.notebook import tqdm

sys.path.insert(0, SHARED)
import smolvla_eval_core as sec
from smolvla_eval_core import (
    RolloutDB, PNP_CONFIG, PNP_RECORDER, PNP_K, BASELINE_STEPS,
    FINAL_STEP_CONFIGS, CAMERAS, IMG_SIZE,
    init_libero_benchmark, build_final_episodes, coverage_report,
    build_pnp_queue, flush_db_to_drive, run_episode_pnp,
)
from libero.libero.envs import OffScreenRenderEnv

sec.init_libero_benchmark()
FINAL_EPISODES = build_final_episodes(PI05_V2_DB)
DB = RolloutDB(SMOLVLA_DB)
policy, preprocess, postprocess = sec.load_smolvla_session(video_dir=SMOLVLA_VIDEO_DIR)


---
## 4. Coverage report (what's left)


In [ ]:
cov_df = coverage_report(DB, FINAL_EPISODES)
if '[4, 5]' in cov_df['step_config'].values:
    missing_45 = int(cov_df.loc[cov_df['step_config'] == '[4, 5]', 'missing'].iloc[0])
else:
    missing_45 = 80
print(f'\nMissing pnp [4,5]: {missing_45} episodes (~{missing_45 * 5.5 / 60:.1f} h at 5.5 min/ep)')


---
## 5. Run missing P&P rollouts (time-budgeted)

**Edit these knobs**, then run and leave the notebook open.

| Knob | Default | Notes |
|------|---------|-------|
| `MAX_RUNTIME_HOURS` | 5.5 | Stop before Colab disconnects |
| `STEP_CONFIG_PRIORITY` | `[4,5]` first | Matches π0.5 best detector config |
| `SYNC_EVERY` | 5 | Flush DB to Drive every N new episodes |
| `SAVE_VIDEO` | `False` | Set `'failures_only'` to debug |


In [ ]:
# ── Run config ───────────────────────────────────────────────────────────────
MAX_RUNTIME_HOURS = 5.5
STEP_CONFIG_PRIORITY = [(4, 5), (3, 4), (2, 3)]
SYNC_EVERY = 5
SAVE_VIDEO = False
SKIP_COMPLETED = True

deadline = time.time() + MAX_RUNTIME_HOURS * 3600
queue = build_pnp_queue(DB, FINAL_EPISODES, STEP_CONFIG_PRIORITY, skip_completed=SKIP_COMPLETED)
print(f'Queue: {len(queue)} missing pnp_uncertainty_only rollouts')
if queue:
    by_cfg = {}
    for item in queue:
        by_cfg[item['step_key']] = by_cfg.get(item['step_key'], 0) + 1
    for k, v in sorted(by_cfg.items()):
        print(f'  {k}: {v}')

method = 'pnp_uncertainty_only'
n_run, n_err = 0, 0
t_start = time.time()

env = None
cur_task = None

for item in queue:
    if time.time() >= deadline:
        print(f'\nTime budget reached ({MAX_RUNTIME_HOURS}h). Stopping.')
        break

    step_indices = item['step_indices']
    ep = item['episode']
    task_key = (ep['suite'], ep['task_idx'])
    if task_key != cur_task:
        if env is not None:
            env.close()
        env = OffScreenRenderEnv(
            bddl_file_name=ep['bddl_path'], camera_names=CAMERAS,
            camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
            has_offscreen_renderer=True, use_camera_obs=True,
            has_renderer=False, reward_shaping=False,
        )
        cur_task = task_key

    remaining_h = (deadline - time.time()) / 3600
    print(f'\n[{n_run+1}/{len(queue)}] {ep["suite"]} T{ep["task_idx"]} ep{ep["ep_idx"]} '
          f'config {step_indices} ({remaining_h:.2f}h left)')

    PNP_CONFIG.enabled = True
    PNP_CONFIG.mode = 'uncertainty'
    PNP_CONFIG.step_indices = step_indices
    PNP_CONFIG.num_iterations = PNP_K
    PNP_CONFIG.time_min = None
    PNP_RECORDER.reset()

    try:
        success, n_steps, elapsed = run_episode_pnp(
            env, ep['init_state'], policy, ep['task_desc'], ep['max_steps'], sec.device,
            suite=ep['suite'], task_idx=ep['task_idx'], episode_idx=ep['ep_idx'],
            db=DB, save_video=SAVE_VIDEO, method=method, final_eval_slice=1,
            num_inference_steps=BASELINE_STEPS,
        )
        n_run += 1
        print(f'  -> success={success} steps={n_steps} elapsed={elapsed/60:.1f}min')
        if n_run % SYNC_EVERY == 0:
            flush_db_to_drive(DB, SMOLVLA_DB)
    except Exception as e:
        n_err += 1
        print(f'  !! ERROR: {e}')
        flush_db_to_drive(DB, SMOLVLA_DB)

if env is not None:
    env.close()

elapsed_h = (time.time() - t_start) / 3600
print(f'\nDone: {n_run} new, {n_err} errors, {elapsed_h:.2f}h elapsed')
flush_db_to_drive(DB, SMOLVLA_DB)
DB.summary()


---
## 6. Verify on Drive


In [ ]:
import sqlite3, pandas as pd

con = sqlite3.connect(SMOLVLA_DB)
summary = pd.read_sql(
    '''SELECT method, pnp_step_indices AS step_cfg,
              COUNT(*) AS n,
              ROUND(AVG(success)*100, 1) AS sr_pct,
              ROUND(AVG(u_mean_episode), 4) AS u_mean
       FROM rollouts
       WHERE final_eval_slice = 1 AND policy_model = 'smolvla'
       GROUP BY method, pnp_step_indices
       ORDER BY method, step_cfg''',
    con,
)
con.close()
print('=== SmolVLA on disk ===')
display(summary)

n_pnp_45 = 0
if not summary.empty:
    row = summary[(summary['method'] == 'pnp_uncertainty_only') & (summary['step_cfg'] == '[4, 5]')]
    if len(row):
        n_pnp_45 = int(row['n'].iloc[0])
print(f'\npnp [4,5] episodes: {n_pnp_45}/80')
if n_pnp_45 >= 60:
    print('Enough for meaningful detector analysis — run pnp_smolvla_jennifer_analysis.ipynb')
elif n_pnp_45 >= 40:
    print('Partial but improved — analysis will be stronger than n=16')
else:
    print('Still thin — consider another run or extend MAX_RUNTIME_HOURS')
